# Modelo Predictivo de Demanda — NYC TLC

**Objetivo:** predecir la demanda diaria de viajes (yellow ta como guíaxi) a 90 días usando Facebook Prophet, con evaluación cuantitativa del error antes de generar la predicción final.

## Setup

Configuración inicial: sesión de Spark y silenciado de logs verbosos de `cmdstanpy` (motor de optimización interno de Prophet). Estos logs son informativos, no errores, pero por defecto son muy detallados.


In [1]:
!pip install plotly --quiet

In [1]:
import sys
if '/home/jovyan/src' not in sys.path:
    sys.path.append('/home/jovyan/src')

import logging
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("prophet").setLevel(logging.WARNING)

import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error

from spark_utils import crear_spark_session

spark = crear_spark_session()

RUTA_SERIE_TIEMPO = "/home/jovyan/data/gold/predictivo/features_serie_tiempo/tipo_vehiculo=yellow/*/*.parquet"
RUTA_DESTINO_GOLD = "/home/jovyan/data/gold/predictivo/forecast_demanda"
HORIZONTE_DIAS = 90
TIPO_VEHICULO = "yellow"

print("Completo")

2026-07-17 08:22:39,909 [INFO] SparkSession creada: 3.5.0


Completo


In [4]:
import sys
if '/home/jovyan/src' not in sys.path:
    sys.path.append('/home/jovyan/src')

from spark_utils import crear_spark_session
from gold.predictivo import construir_features_serie_tiempo

spark_build = crear_spark_session()
construir_features_serie_tiempo(spark_build)
spark_build.stop()

2026-07-17 08:11:50,069 [INFO] SparkSession creada: 3.5.0
2026-07-17 08:12:07,858 [INFO] Gold escrito (overwrite): predictivo/features_serie_tiempo (365 filas)
2026-07-17 08:12:07,892 [INFO]   features_serie_tiempo: yellow 2023 listo
2026-07-17 08:12:14,324 [INFO] Gold escrito (append): predictivo/features_serie_tiempo (366 filas)
2026-07-17 08:12:14,327 [INFO]   features_serie_tiempo: yellow 2024 listo
2026-07-17 08:12:21,322 [INFO] Gold escrito (append): predictivo/features_serie_tiempo (365 filas)
2026-07-17 08:12:21,326 [INFO]   features_serie_tiempo: yellow 2025 listo
2026-07-17 08:12:22,239 [INFO] Gold escrito (append): predictivo/features_serie_tiempo (365 filas)
2026-07-17 08:12:22,246 [INFO]   features_serie_tiempo: green 2023 listo
2026-07-17 08:12:23,111 [INFO] Gold escrito (append): predictivo/features_serie_tiempo (366 filas)
2026-07-17 08:12:23,122 [INFO]   features_serie_tiempo: green 2024 listo
2026-07-17 08:12:23,820 [INFO] Gold escrito (append): predictivo/features_se

## Fase 0 — Auditoría de calidad de datos

Antes de entrenar cualquier modelo, se inspecciona la serie completa en busca de valores extremos que puedan distorsionar el ajuste. Se calculan estadísticos descriptivos y se listan los días de menor volumen.


In [2]:
df_spark = spark.read.parquet(RUTA_SERIE_TIEMPO).orderBy("fecha")

df_spark.select(
    F.min("total_viajes").alias("min_viajes"),
    F.max("total_viajes").alias("max_viajes"),
    F.avg("total_viajes").alias("promedio"),
    F.stddev("total_viajes").alias("desviacion")
).show()

# Los 10 días con menor volumen — para identificar outliers
df_spark.orderBy("total_viajes").select("fecha", "total_viajes", "dia_semana").show(10)

+----------+----------+------------------+------------------+
|min_viajes|max_viajes|          promedio|        desviacion|
+----------+----------+------------------+------------------+
|      2567|    175148|111232.72536496351|20118.967308853997|
+----------+----------+------------------+------------------+

+----------+------------+----------+
|     fecha|total_viajes|dia_semana|
+----------+------------+----------+
|2023-09-22|        2567|         6|
|2023-09-24|        2985|         1|
|2023-09-23|        3473|         7|
|2023-09-21|       38856|         5|
|2023-12-25|       43891|         2|
|2024-12-25|       53236|         4|
|2023-07-04|       56277|         3|
|2024-07-04|       60048|         5|
|2023-11-23|       60448|         5|
|2023-07-03|       62960|         2|
+----------+------------+----------+
only showing top 10 rows



In [3]:
FECHAS_EXCLUIR = ['2023-09-22', '2023-09-23', '2023-09-24']
print(f"Fechas excluidas por falla de ingesta confirmada: {FECHAS_EXCLUIR}")

Fechas excluidas por falla de ingesta confirmada: ['2023-09-22', '2023-09-23', '2023-09-24']


## Fase 1 — Train / Test Split temporal

Al tratarse de una serie de tiempo, el split debe ser **cronológico**, no aleatorio: un split aleatorio filtraría información del futuro hacia el entrenamiento (data leakage). Se reservan los últimos 90 días como conjunto de pruebro.


In [4]:
corte = df_spark.select(
    F.date_sub(F.max("fecha"), HORIZONTE_DIAS).alias("corte")
).collect()[0][0]

train_spark = df_spark.filter(F.col("fecha") <= corte)
test_spark = df_spark.filter(F.col("fecha") > corte)

train_pd = (
    train_spark.select(F.col("fecha").alias("ds"), F.col("total_viajes").alias("y"))
    .orderBy("ds")
    .toPandas()
)
train_pd['ds'] = train_pd['ds'].astype('datetime64[ns]')
train_pd = train_pd[~train_pd['ds'].astype(str).isin(FECHAS_EXCLUIR)].reset_index(drop=True)

test_pd = (
    test_spark.select(F.col("fecha").alias("ds"), F.col("total_viajes").alias("y"))
    .orderBy("ds")
    .toPandas()
)
test_pd['ds'] = test_pd['ds'].astype('datetime64[ns]')

print(f"Train: {len(train_pd)} días ({train_pd['ds'].min().date()} a {train_pd['ds'].max().date()})")
print(f"Test:  {len(test_pd)} días ({test_pd['ds'].min().date()} a {test_pd['ds'].max().date()})")

Train: 1003 días (2023-01-01 a 2025-10-02)
Test:  90 días (2025-10-03 a 2025-12-31)


## Fase 2 — Modelo baseline

Se entrena un primer modelo Prophet con estacionalidad anual y semanal (sin estacionalidad diaria, ya que los datos están agregados a nivel de día.

In [5]:
modelo_baseline = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
    interval_width=0.95
)
modelo_baseline.fit(train_pd)

futuro_baseline = modelo_baseline.make_future_dataframe(periods=HORIZONTE_DIAS)
forecast_baseline = modelo_baseline.predict(futuro_baseline)

print("Modelo baseline entrenado")
forecast_baseline[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(5)

Modelo baseline entrenado


,ds,yhat,yhat_lower,yhat_upper
1088,2025-12-27,118174.260715,98670.240437,138164.463428
1089,2025-12-28,98877.942033,79152.082709,117395.111983
1090,2025-12-29,92547.327778,73424.439200,112240.261411
1091,2025-12-30,106083.308045,86329.341965,126086.686760
1092,2025-12-31,111788.822014,91478.505963,129828.678843


## Fase 3 — Evaluación cuantitativa (baseline)

Se compara la predicción contra los valores reales del conjunto de prueba usando tres métricas estándar:

- **MAE** (Error Absoluto Medio): magnitud promedio del error, en viajes.
- **MAPE** (Error Porcentual Absoluto Medio): el más interpretable — "el modelo se equivoca en promedio X%". <10% se considera bueno, 10–20% aceptable, >20% amerita revisión.
- **RMSE** (Raíz del Error Cuadrático Medio): penaliza más los errores grandes; útil para detectar si hay días puntuales con fallos importantes.

In [6]:
def evaluar(forecast, test, nombre):
    comparacion = forecast[['ds', 'yhat']].merge(test[['ds', 'y']], on='ds')
    mae = mean_absolute_error(comparacion['y'], comparacion['yhat'])
    mape = mean_absolute_percentage_error(comparacion['y'], comparacion['yhat']) * 100
    rmse = np.sqrt(mean_squared_error(comparacion['y'], comparacion['yhat']))
    print(f"[{nombre}] MAE: {mae:,.0f} | MAPE: {mape:.2f}% | RMSE: {rmse:,.0f}")
    return {"modelo": nombre, "mae": mae, "mape": mape, "rmse": rmse}

metricas_baseline = evaluar(forecast_baseline, test_pd, "Baseline")

[Baseline] MAE: 12,352 | MAPE: 11.28% | RMSE: 17,687


## Fase 4 — Mejora: feriados como regresor

La demanda de taxis en NYC cambia notablemente en feriados (Navidad, Año Nuevo, Thanksgiving, 4 de julio). Prophet permite incorporar el calendario de feriados de un país directamente como regresor, sin necesidad de features manuales.

Se entrena un segundo modelo con esta mejora y se compara contra el baseline usando las mismas métricas.

In [7]:
modelo_holidays = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
    interval_width=0.95
)
modelo_holidays.add_country_holidays(country_name='US')
modelo_holidays.fit(train_pd)

futuro_holidays = modelo_holidays.make_future_dataframe(periods=HORIZONTE_DIAS)
forecast_holidays = modelo_holidays.predict(futuro_holidays)

metricas_holidays = evaluar(forecast_holidays, test_pd, "Con feriados")

[Con feriados] MAE: 11,334 | MAPE: 9.97% | RMSE: 15,450


### Comparación de Métricas

| Métrica | Baseline | Con feriados | Mejora |
| :--- | :--- | :--- | :--- |
| **MAE** | 13.352 | 11.334 | -15.11% |
| **MAPE** | 11.28% | 9.97% | -1.31 pp |
| **RMSE** | 17.687 | 15.450 | -12.65% |

Las tres métricas mejoran de forma consistente. El RMSE muestra una reducción significativa, lo que confirma que el modelo ha logrado mitigar los errores grandes que anteriormente se concentraban en los días feriados.8% |

## Fase 4.1 — Comparación con un modelo alternativo: SARIMA

Como ejercicio de comparación metodológica, se evalúa un segundo enfoque de forecasting — SARIMA — sobre el mismo split de entrenamiento/prueba usado con Prophet. El objetivo es contrastar dos filosofías distintas de modelado de series de tiempo:

- **Prophet**: modelo aditivo con estacionalidad múltiple (semanal + anual) y feriados como regresor explícito.
- **SARIMA**: modelo autoregresivo clásico, aquí configurado únicamente con estacionalidad semanal (`m=7`), sin variables exógenas.

Los parámetros óptimos de SARIMA se buscan automáticamente con `auto_arima` (búsqueda stepwise, minimizando AIC) en vez de fijarlos a mano.


In [13]:
import pmdarima as pm

serie_train = train_pd.set_index('ds')['y']
serie_test = test_pd.set_index('ds')['y']

modelo_sarima = pm.auto_arima(
    serie_train,
    seasonal=True,
    m=7,                    # estacionalidad semanal
    d=None,
    D=None,
    trace=True,
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True,
    max_p=3, max_q=3, max_P=2, max_Q=2
)

print(modelo_sarima.summary())

Performing stepwise search to minimize aic
 ARIMA(2,1,2)(1,0,1)[7] intercept   : AIC=inf, Time=2.45 sec
 ARIMA(0,1,0)(0,0,0)[7] intercept   : AIC=21888.975, Time=0.02 sec
 ARIMA(1,1,0)(1,0,0)[7] intercept   : AIC=21549.844, Time=0.14 sec
 ARIMA(0,1,1)(0,0,1)[7] intercept   : AIC=21678.515, Time=0.23 sec
 ARIMA(0,1,0)(0,0,0)[7]             : AIC=21886.988, Time=0.02 sec
 ARIMA(1,1,0)(0,0,0)[7] intercept   : AIC=21889.184, Time=0.04 sec
 ARIMA(1,1,0)(2,0,0)[7] intercept   : AIC=21349.863, Time=0.87 sec
 ARIMA(1,1,0)(2,0,1)[7] intercept   : AIC=21365.012, Time=1.89 sec
 ARIMA(1,1,0)(1,0,1)[7] intercept   : AIC=inf, Time=1.43 sec
 ARIMA(0,1,0)(2,0,0)[7] intercept   : AIC=21474.655, Time=0.21 sec
 ARIMA(2,1,0)(2,0,0)[7] intercept   : AIC=21464.819, Time=0.41 sec
 ARIMA(1,1,1)(2,0,0)[7] intercept   : AIC=inf, Time=1.87 sec
 ARIMA(0,1,1)(2,0,0)[7] intercept   : AIC=21349.848, Time=0.97 sec
 ARIMA(0,1,1)(1,0,0)[7] intercept   : AIC=21472.702, Time=0.34 sec
 ARIMA(0,1,1)(2,0,1)[7] intercept   :

In [14]:
n_periods = len(serie_test)
forecast_sarima, conf_int = modelo_sarima.predict(n_periods=n_periods, return_conf_int=True)

comparacion_sarima = pd.DataFrame({
    'ds': serie_test.index,
    'y': serie_test.values,
    'yhat': forecast_sarima.values
})

mae_sarima = mean_absolute_error(comparacion_sarima['y'], comparacion_sarima['yhat'])
mape_sarima = mean_absolute_percentage_error(comparacion_sarima['y'], comparacion_sarima['yhat']) * 100
rmse_sarima = np.sqrt(mean_squared_error(comparacion_sarima['y'], comparacion_sarima['yhat']))

metricas_sarima = {"modelo": "SARIMA", "mae": mae_sarima, "mape": mape_sarima, "rmse": rmse_sarima}
print(f"[SARIMA] MAE: {mae_sarima:,.0f} | MAPE: {mape_sarima:.2f}% | RMSE: {rmse_sarima:,.0f}")

[SARIMA] MAE: 23,273 | MAPE: 18.08% | RMSE: 27,111


### Resultado: Prophet vs. SARIMA

| Modelo | MAE | MAPE | RMSE |
| :--- | :--- | :--- | :--- |
| Prophet (baseline) | 12,249 | 11.23% | 17,455 |
| **Prophet (con feriados)** | **11,334** | **9.97%** | **15,450** |
| SARIMA | 23,273 | 18.08% | 27,111 |

**Prophet supera claramente a SARIMA** (MAPE de 9.97% vs. 18.08% — prácticamente la mitad de error). La causa se explica revisando el modelo ganador de `auto_arima`: `SARIMAX(0,1,0)(1,0,[1,2])[7]`. La componente no estacional `(0,1,0)` equivale esencialmente a un *random walk* (el valor de mañana se explica por el de hoy más ruido), y la única estructura capturada es la estacionalidad semanal — el modelo no tiene forma de anticipar feriados, picos de fin de año, ni el patrón anual completo.

Esto se confirma con la curtosis de los residuos del modelo (**15.86**, muy por encima del 3 esperado en una distribución normal): indica errores grandes y ocasionales que el modelo no logra explicar, consistentes con días atípicos (feriados, eventos especiales) que SARIMA, en esta configuración, no puede anticipar.

## Fase 5 — Reentrenamiento final con el histórico completo

Una vez validado el modelo (Fases 3–4) sobre el conjunto de prueba, se reentrena con el **100% de los datos históricos disponibles** (incluyendo el tramo que antes se usó como test) para generar la predicción real hacia el futuro. Esto maximiza la información disponible para el modelo de producción.

In [8]:
df_completo_pd = (
    df_spark.select(F.col("fecha").alias("ds"), F.col("total_viajes").alias("y"))
    .orderBy("ds")
    .toPandas()
)
df_completo_pd['ds'] = df_completo_pd['ds'].astype('datetime64[ns]')
df_completo_pd = df_completo_pd[~df_completo_pd['ds'].astype(str).isin(FECHAS_EXCLUIR)].reset_index(drop=True)

modelo_produccion = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
    interval_width=0.95
)
modelo_produccion.add_country_holidays(country_name='US')
modelo_produccion.fit(df_completo_pd)

futuro_real = modelo_produccion.make_future_dataframe(periods=HORIZONTE_DIAS)
forecast_real = modelo_produccion.predict(futuro_real)

print(f"Predicción generada hasta: {forecast_real['ds'].max().date()}")
forecast_real[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(5)

Predicción generada hasta: 2026-03-31


,ds,yhat,yhat_lower,yhat_upper
1178,2026-03-27,138785.500286,118950.636849,157039.914532
1179,2026-03-28,141490.176760,122932.611745,161554.945298
1180,2026-03-29,124441.374096,107110.366001,143983.287336
1181,2026-03-30,121375.432525,102547.516390,140691.883781
1182,2026-03-31,134656.683864,116133.585992,153683.159469


## Fase 6 — Exportación a Gold

Se exporta el forecast como tabla Parquet particionada por `tipo_vehiculo`, siguiendo el mismo patrón de las demás tablas Gold del proyecto (`descriptivo.py`, `diagnostico.py`).

La columna `es_prediccion` (booleana) permite en Power BI diferenciar visualmente la serie histórica de la proyección futura dentro del mismo visual.


In [9]:
forecast_export = forecast_real[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
forecast_export = forecast_export.rename(columns={
    'ds': 'fecha',
    'yhat': 'prediccion',
    'yhat_lower': 'prediccion_min',
    'yhat_upper': 'prediccion_max'
})

fecha_max_historica = df_completo_pd['ds'].max()
forecast_export['es_prediccion'] = forecast_export['fecha'] > fecha_max_historica
forecast_export['tipo_vehiculo'] = TIPO_VEHICULO

print(f"Fecha máxima histórica: {fecha_max_historica.date()}")
print(f"Filas históricas: {(~forecast_export['es_prediccion']).sum()}")
print(f"Filas de predicción: {forecast_export['es_prediccion'].sum()}")

forecast_spark = spark.createDataFrame(forecast_export)

forecast_spark.write.mode("overwrite").partitionBy("tipo_vehiculo").parquet(RUTA_DESTINO_GOLD)

print(f"Forecast exportado: {forecast_spark.count()} filas -> {RUTA_DESTINO_GOLD}")

Fecha máxima histórica: 2025-12-31
Filas históricas: 1093
Filas de predicción: 90
Forecast exportado: 1183 filas -> /home/jovyan/data/gold/predictivo/forecast_demanda


## Proceso de los demás tipos de Vehículos

In [11]:
RUTA_BASE = "/home/jovyan/data/gold/predictivo/features_serie_tiempo/tipo_vehiculo={tipo}/"

def entrenar_y_exportar(spark, tipo, fechas_excluir=None, usar_feriados=True):
    fechas_excluir = fechas_excluir or []
    print(f"\n=== {tipo.upper()} ===")
    
    df_spark = spark.read.parquet(RUTA_BASE.format(tipo=tipo)).orderBy("fecha")
    
    df_completo_pd = (
        df_spark.select(F.col("fecha").alias("ds"), F.col("total_viajes").alias("y"))
        .orderBy("ds").toPandas()
    )
    df_completo_pd['ds'] = df_completo_pd['ds'].astype('datetime64[ns]')
    
    if fechas_excluir:
        df_completo_pd = df_completo_pd[~df_completo_pd['ds'].astype(str).isin(fechas_excluir)].reset_index(drop=True)
    
    modelo_final = Prophet(yearly_seasonality=True, weekly_seasonality=True,
                           daily_seasonality=False, changepoint_prior_scale=0.05, interval_width=0.95)
    
    if usar_feriados:
        modelo_final.add_country_holidays(country_name='US')
        
    modelo_final.fit(df_completo_pd)
    forecast_final = modelo_final.predict(modelo_final.make_future_dataframe(periods=HORIZONTE_DIAS))
    
    forecast_export = forecast_final[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
    forecast_export = forecast_export.rename(columns={
        'ds': 'fecha',
        'yhat': 'prediccion',
        'yhat_lower': 'prediccion_min', 
        'yhat_upper': 'prediccion_max'
    })
    
    fecha_max = df_completo_pd['ds'].max()
    forecast_export['es_prediccion'] = forecast_export['fecha'] > fecha_max
    forecast_export['tipo_vehiculo'] = tipo
    
    forecast_spark = spark.createDataFrame(forecast_export)
    forecast_spark.write.mode("append").partitionBy("tipo_vehiculo").parquet(RUTA_DESTINO_GOLD)
    print(f"  Exportado: {forecast_spark.count()} filas -> tipo_vehiculo={tipo}")

# Ejecutar para los 3 tipos
for tipo in ["green", "fhv", "fhvhv"]:
    entrenar_y_exportar(spark, tipo, fechas_excluir=None, usar_feriados=True)


=== GREEN ===
  Exportado: 1186 filas -> tipo_vehiculo=green

=== FHV ===
  Exportado: 1186 filas -> tipo_vehiculo=fhv

=== FHVHV ===
  Exportado: 1186 filas -> tipo_vehiculo=fhvhv


In [17]:
pip install pmdarima --break-system-packages

IOStream.flush timed out
Note: you may need to restart the kernel to use updated packages.
